<a href="https://colab.research.google.com/github/gabrielcarcedo/Multimodal-GNN-for-Chagas-Classification/blob/main/VGAE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Multimodal Graph Learning for Chagas Disease Classification**
## *Data Augmentation via Variational Graph Autoencoder (VGAE)*

**Authors:** Gabriel Carcedo-Rodríguez, Erik Molino-Minero-Re, Jorge Perez-Gonzalez, Nidiyare Hevia-Montiel.

**Description:**
This notebook contains the implementation of the generative phase of our proposed framework. It focuses on training a Variational Graph Autoencoder (VGAE) using multimodal clinical data (ECG, ECHO, Doppler, ELISA) after feature selection. The goal of this model is to learn the underlying topological distribution of the biological features and generate highly realistic synthetic subjects. This targeted data augmentation strategy is designed to overcome the "Small Data" limitation, prevent model overfitting, and enrich the dataset for subsequent robust classification tasks.

# **Install dependencies**

In [ ]:
!pip install torch-geometric

# **Import Packages and Libraries**

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import label_binarize

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Usando dispositivo: cpu


# **Preprocess Data**

In [ ]:
db = pd.read_csv('bd_final_multimodal_120821.csv')
db.set_index('ID paciente', inplace=True)
db.dropna(inplace=True)
db = db.drop(columns=['lgGT control', 'lgG1 control', 'lgG2a control'])

echo_features = ['LVd']
db_echo = db[echo_features]
db_echo = StandardScaler().fit_transform(db_echo)
db_echo = pd.DataFrame(db_echo, columns=echo_features)

elisa_features = ['lgGT','lgG1','lgG2a']
db_elisa = db[elisa_features]
db_elisa = StandardScaler().fit_transform(db_elisa)
db_elisa = pd.DataFrame(db_elisa, columns=elisa_features)

ecg_features = ['HR', 'HRV', 'CV', 'QT Interval', 'ST Interval', 'QTc', 'SR mean']
db_ecg = db[ecg_features]
db_ecg = StandardScaler().fit_transform(db_ecg)
db_ecg = pd.DataFrame(db_ecg, columns=ecg_features)

doppler_features = ['AO Pre-ejection time Avg', 'AO Ejection Time SD', 'AO Peak Acceleration Avg', 'MV HR Avg']
db_doppler = db[doppler_features]
db_doppler = StandardScaler().fit_transform(db_doppler)
db_doppler = pd.DataFrame(db_doppler, columns=doppler_features)

db_all = pd.concat([db_echo, db_elisa, db_ecg, db_doppler], axis=1)
db_all['Day'] = db['Day'].values
db_all['Label'] = db['Label'].values
db_all.index = db.index

db_echo = db_all[echo_features + ['Label']]
db_elisa = db_all[elisa_features + ['Label']]
db_ecg = db_all[ecg_features + ['Label']]
db_doppler = db_all[doppler_features + ['Label']]

db_echo_sano = db_echo[db_echo['Label']==0]
db_echo_agudo = db_echo[db_echo['Label']==1]
db_echo_cronico = db_echo[db_echo['Label']==2]

db_elisa_sano = db_elisa[db_elisa['Label']==0]
db_elisa_agudo = db_elisa[db_elisa['Label']==1]
db_elisa_cronico = db_elisa[db_elisa['Label']==2]

db_ecg_sano = db_ecg[db_ecg['Label']==0]
db_ecg_agudo = db_ecg[db_ecg['Label']==1]
db_ecg_cronico = db_ecg[db_ecg['Label']==2]

db_doppler_sano = db_doppler[db_doppler['Label']==0]
db_doppler_agudo = db_doppler[db_doppler['Label']==1]
db_doppler_cronico = db_doppler[db_doppler['Label']==2]

datos_sintéticos_echo = pd.DataFrame(columns=echo_features + ['Label'])
datos_sintéticos_elisa = pd.DataFrame(columns=elisa_features + ['Label'])
datos_sintéticos_ecg = pd.DataFrame(columns=ecg_features + ['Label'])
datos_sintéticos_doppler = pd.DataFrame(columns=doppler_features + ['Label'])

# **Variational Graph Auntoencoder (VGAE)** Model

In [ ]:
class GraphEncoder(nn.Module):
    def __init__(self, in_channels, gcn_hidden, graph_emb_dim):
        super().__init__()
        self.conv1 = GCNConv(in_channels, gcn_hidden)
        self.conv2 = GCNConv(gcn_hidden, gcn_hidden)
        self.post_pool = nn.Linear(gcn_hidden, gcn_hidden)
        self.lin_mu = nn.Linear(gcn_hidden, graph_emb_dim)
        self.lin_logstd = nn.Linear(gcn_hidden, graph_emb_dim)

    def forward(self, x, edge_index, batch):
        h = F.relu(self.conv1(x, edge_index))
        h = F.relu(self.conv2(h, edge_index))
        h_graph = global_mean_pool(h, batch)    # [B, gcn_hidden]
        h_graph = F.relu(self.post_pool(h_graph))
        mu = self.lin_mu(h_graph)
        logstd = self.lin_logstd(h_graph)  # log(sigma)
        return mu, logstd

class GraphDecoder(nn.Module):
    def __init__(self, graph_emb_dim, hidden_dim, out_features, p_drop=0.2):
        super().__init__()
        self.fc1 = nn.Linear(graph_emb_dim, hidden_dim)
        self.ln1 = nn.LayerNorm(hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.ln2 = nn.LayerNorm(hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, out_features)
        self.p_drop = p_drop

    def forward(self, z):
        h = F.relu(self.ln1(self.fc1(z)))
        h = F.dropout(h, p=self.p_drop, training=self.training)
        h = F.relu(self.ln2(self.fc2(h)))
        h = F.dropout(h, p=self.p_drop, training=self.training)
        x_row = self.fc3(h)
        return x_row

## Graphs Cronstruction

In [ ]:
def create_graphs_tabular(df, corr_by_class, feature_cols, label_col='Label', threshold=0.5):
    graphs = []
    for idx, row in df.iterrows():
        label = int(row[label_col])
        row_features = row[feature_cols].astype(float).values
        x = torch.tensor(row_features, dtype=torch.float).view(-1, 1)  # [n_features, 1]
        if label not in corr_by_class:
            raise KeyError(f"Clase {label} no encontrada en corr_by_class")
        edge_index = create_edges_from_MI(corr_by_class[label], threshold=threshold)
        y = torch.tensor(label, dtype=torch.long)
        g = Data(x=x, edge_index=edge_index, y=y)
        graphs.append(g)
    return graphs

def create_edges_from_MI(MI_matrix, threshold=0.5):
    edge_index = []
    n = MI_matrix.shape[0]
    for i in range(n):
        for j in range(i+1, n):
            if abs(MI_matrix.iloc[i, j]) >= threshold:
                edge_index.append([i, j])
                edge_index.append([j, i])
    if len(edge_index) == 0:
        return torch.empty((2, 0), dtype=torch.long)
    return torch.tensor(edge_index, dtype=torch.long).t().contiguous()

## **VAGE** Train Function

In [ ]:
def train_epoch(encoder, decoder, device, loader, optimizer, epoch, warmup_epochs=50, beta_max=0.1):
    encoder.train(); decoder.train()
    total_loss = 0.0
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()

        mu, logstd = encoder(data.x, data.edge_index, data.batch)   # [B, z_dim]
        z = reparametrize(mu, logstd)                               # [B, z_dim]
        x_recon = decoder(z)                                        # [B, n_features]

        batch_size = int(data.batch.max().item()) + 1
        rows = []
        for i in range(batch_size):
            mask = (data.batch == i)
            nodes_feats = data.x[mask].view(-1)  # [n_features]
            rows.append(nodes_feats.unsqueeze(0))
        x_target = torch.cat(rows, dim=0).to(device)  # [B, n_features]

        node_loss = F.mse_loss(x_recon, x_target)
        kl = kl_loss(mu, logstd)
        beta = min(beta_max, (epoch + 1) / float(warmup_epochs) * beta_max)  # warm-up to beta_max
        loss = node_loss + beta * kl

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)

## Reparametrize: Util Function

In [ ]:
def reparametrize(mu, logstd):
    std = torch.exp(logstd)
    eps = torch.randn_like(std)
    return mu + eps * std

## **Kullback-Leibler** Loss Function

In [ ]:
def kl_loss(mu, logstd):
    kl_per_sample = -0.5 * torch.sum(1 + 2*logstd - mu.pow(2) - (2*logstd).exp(), dim=1)
    return kl_per_sample.mean()

## Evaluate Reconstruction

In [ ]:
def evaluate_reconstruction(encoder, decoder, device, loader):
    encoder.eval(); decoder.eval()
    all_recons = []
    all_targets = []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            mu, logstd = encoder(data.x, data.edge_index, data.batch)
            z = mu  # deterministic
            x_recon = decoder(z)

            batch_size = int(data.batch.max().item()) + 1
            rows = []
            for i in range(batch_size):
                mask = (data.batch == i)
                nodes_feats = data.x[mask].view(-1)
                rows.append(nodes_feats.unsqueeze(0))
            x_target = torch.cat(rows, dim=0).to(device)

            all_recons.append(x_recon.cpu().numpy())
            all_targets.append(x_target.cpu().numpy())
    if len(all_recons) == 0:
        return np.empty((0,)), np.empty((0,))
    return np.concatenate(all_recons, axis=0), np.concatenate(all_targets, axis=0)

## **Training** *Example*

In [ ]:
label_col = 'Label'
feature_cols = features
modalidad = 'ECHO'
clase = 'Sano'

corr_by_class = {}
for lab in sorted(df_train[label_col].unique()):
    df_class = df_train[df_train[label_col] == lab]
    corr = df_class[feature_cols].corr()
    corr_by_class[int(lab)] = corr

# Graphs construction
train_graphs = create_graphs_tabular(df_train, corr_by_class, feature_cols, label_col=label_col, threshold=0.5)
test_graphs  = create_graphs_tabular(df_test,  corr_by_class, feature_cols, label_col=label_col, threshold=0.5)

# DataLoaders
batch_size = 8
train_loader = DataLoader(train_graphs, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_graphs,  batch_size=batch_size, shuffle=False)

# Hyperparameters
in_channels = 1
gcn_hidden  = 1024
z_dim       = 16
decoder_hidden = 1024
n_features = len(feature_cols)
epochs = 50
best_val_mse = np.inf
patience = 10
pat_cnt = 0

encoder = GraphEncoder(in_channels=in_channels, gcn_hidden=gcn_hidden, graph_emb_dim=z_dim).to(device)
decoder = GraphDecoder(graph_emb_dim=z_dim, hidden_dim=decoder_hidden, out_features=n_features, p_drop=0.2).to(device)

optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=10, min_lr=1e-6)

train_losses, val_losses = [], []

for epoch in range(epochs):
    train_loss = train_epoch(encoder, decoder, device, train_loader, optimizer, epoch, warmup_epochs=50, beta_max=0.1)
    recons, targets = evaluate_reconstruction(encoder, decoder, device, test_loader)
    val_mse = ((recons - targets)**2).mean() if recons.size else np.inf
    scheduler.step(val_mse)
    print(f'Epoch {epoch+1:03d}  TrainLoss={train_loss:.6f}  ValMSE={val_mse:.8f}')

    if val_mse < best_val_mse:
        best_val_mse = val_mse
        torch.save({'encoder': encoder.state_dict(), 'decoder': decoder.state_dict()}, os.path.join(folder, f'{modalidad}_{clase}_best_model.pth'))
        pat_cnt = 0
    else:
        pat_cnt += 1
        if pat_cnt >= patience:
            print("Early stopping")
            break

    train_losses.append(train_loss)
    val_losses.append(val_mse)

# Visulize Losses
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Train Loss', lw=3, alpha=0.75, color='cyan')
plt.plot(val_losses, label='Val MSE', lw=3, alpha=0.75, color='darkblue')
plt.xlabel('Epoch')
plt.ylabel('Value')
plt.title('Training and Validation Losses')
plt.grid(ls='-.', lw=1, alpha=0.5, color='lightgray')
plt.legend()

## **Synthetic Data** Generation

In [ ]:
ck = torch.load(os.path.join(folder, f'{modalidad}_{clase}_best_model.pth'), map_location=device)
encoder.load_state_dict(ck['encoder'])
decoder.load_state_dict(ck['decoder'])
encoder.eval(); decoder.eval()

target_label = 0  # Class
mus = []
stds = []
with torch.no_grad():
    for g in train_graphs:
        if int(g.y.item()) == target_label:
            batch = torch.zeros(g.x.size(0), dtype=torch.long, device=g.x.device)
            mu_g, logstd_g = encoder(g.x.to(device), g.edge_index.to(device), batch)
            mu_arr  = mu_g.cpu().numpy().reshape(-1)           # [z_dim]
            std_arr = torch.exp(logstd_g).cpu().numpy().reshape(-1)
            mus.append(mu_arr)
            stds.append(std_arr)

mus = np.array(mus)    # [n_samples_class, z_dim]
stds = np.array(stds)  # [n_samples_class, z_dim]

if mus.shape[0] == 0:
    raise RuntimeError(f"No hay muestras para la clase {target_label} en train_graphs")

# Generate N instances
N_new = len(df_train[df_train[label_col] == 0])
# N_new = min(14, 2 * mus.shape[0])
idx = np.random.choice(len(mus), size=N_new, replace=True)
mu_sel  = mus[idx]   # [N_new, z_dim]
std_sel = stds[idx]  # [N_new, z_dim]

z_new = torch.tensor(np.random.randn(N_new, z_dim) * std_sel + mu_sel, dtype=torch.float).to(device)
with torch.no_grad():
    x_new = decoder(z_new).cpu().numpy()  # [N_new, n_features]

# DataFrame Construction
new_rows = pd.DataFrame(x_new, columns=feature_cols)
new_rows[label_col] = target_label
nueva_columna = 'ID paciente'

my_list_sano = []
for i in range(len(df_train)):
    my_list_sano.append(f"Sintetico_{clase}_{i:02d}")

new_rows.insert(0, nueva_columna, my_list_sano)
new_rows.set_index(nueva_columna, inplace=True)

datos_sintéticos_echo = pd.concat([datos_sintéticos_echo, new_rows])
datos_sintéticos_echo